# 01 — Merge and clean

**Takes in:** `data/tract_outcomes_simple.csv`, `data/tract_covariates.csv` (written by [`00_pull.ipynb`](00_pull.ipynb))

**Does:** inner-joins tract-level upward-mobility outcomes onto tract-level neighborhood covariates on `(state, county, tract)`, drops tracts with no mobility estimate or no population density, bins population density into five categories, and flags tracts above the national tract median of mobility. Row counts are printed before and after every step that can drop rows.

**Outputs:** `data/analysis_sample.csv` — one row per Census tract


In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("."))
import utils

pd.set_option("display.max_columns", 40)
print("reading from:", utils.DATA_DIR)

reading from: /Users/maxfortner/Documents/Dartmouth/QSS20/qss20-rural-mobility/data


## Functions

In [2]:
def read_outcomes(data_dir):
    """Read the Atlas outcomes table, keeping only the pooled p25 mobility measure."""
    return pd.read_csv(
        os.path.join(data_dir, "tract_outcomes_simple.csv"),
        usecols=utils.ID_COLS + ["cz", "czname", utils.MOBILITY_VAR, utils.COUNT_VAR],
    )


def read_covariates(data_dir):
    """Read the Atlas covariates table.

    `cz` and `czname` are dropped because they also appear in the outcomes
    table; keeping both would produce _x / _y suffixes after the merge.
    """
    cov = pd.read_csv(os.path.join(data_dir, "tract_covariates.csv"))
    return cov.drop(columns=["cz", "czname"])


def merge_with_diagnostics(outcomes, covariates):
    """Inner join on tract id, printing row counts before and after."""
    print("rows before merge -- outcomes: {:,} | covariates: {:,}".format(
        len(outcomes), len(covariates)))
    merged = outcomes.merge(covariates, on=utils.ID_COLS, how="inner",
                            validate="one_to_one")
    print("rows after inner join: {:,}".format(len(merged)))
    print("outcomes rows dropped: {:,} | covariate rows dropped: {:,}".format(
        len(outcomes) - len(merged), len(covariates) - len(merged)))
    return merged


def drop_unusable(df):
    """Drop tracts that cannot be placed on either axis of the analysis."""
    n_start = len(df)
    out = df.dropna(subset=[utils.MOBILITY_VAR, "popdensity2000"])
    print("dropped {:,} tracts missing mobility or density; {:,} remain".format(
        n_start - len(out), len(out)))
    return out


def add_analysis_vars(df):
    """Add the density category and the above-national-median mobility flag."""
    df = df.copy()
    df["density_cat"] = pd.cut(df["popdensity2000"], bins=utils.DENSITY_BINS,
                               labels=utils.DENSITY_LABELS, right=False)
    national_median = df[utils.MOBILITY_VAR].median()
    df["above_median_mobility"] = df[utils.MOBILITY_VAR] > national_median
    print("national tract median of {}: {:.3f}".format(
        utils.MOBILITY_VAR, national_median))
    return df

## Read

In [3]:
outcomes = read_outcomes(utils.DATA_DIR)
covariates = read_covariates(utils.DATA_DIR)

print("outcomes:  {:,} rows x {} cols".format(*outcomes.shape))
print("covariates: {:,} rows x {} cols".format(*covariates.shape))

outcomes:  73,278 rows x 7 cols
covariates: 74,044 rows x 36 cols


## Merge

In [4]:
merged_raw = merge_with_diagnostics(outcomes, covariates)
merged = drop_unusable(merged_raw)


rows before merge -- outcomes: 73,278 | covariates: 74,044
rows after inner join: 73,199
outcomes rows dropped: 79 | covariate rows dropped: 845
dropped 1,241 tracts missing mobility or density; 71,958 remain


The join is one-to-one on `(state, county, tract)` and `validate="one_to_one"` makes pandas raise if that ever stops being true. The tracts lost at the inner join are ones that appear in only one of the two Atlas tables; the tracts lost at the next step are ones with a suppressed mobility estimate (too few children) or no 2000 density.


## Derive the analysis variables

In [5]:
analysis = add_analysis_vars(merged)

print("\ntracts per density category:")
print(analysis["density_cat"].value_counts().sort_index().to_string())

national tract median of kfr_pooled_pooled_p25: 0.425

tracts per density category:
density_cat
Rural\n(<100)            17978
Small town\n(100-500)    12777
Suburban\n(500-2k)       23101
Urban\n(2k-10k)          15520
Dense urban\n(10k+)       2582


In [6]:
rural = analysis[analysis["popdensity2000"] < utils.RURAL_CUTOFF]
print("rural tracts (<{} people/sq. mi.): {:,}".format(utils.RURAL_CUTOFF, len(rural)))
print("  of which above the national median: {:,} ({:.1%})".format(
    int(rural["above_median_mobility"].sum()),
    rural["above_median_mobility"].mean()))

rural tracts (<100 people/sq. mi.): 17,978
  of which above the national median: 9,087 (50.5%)


## Missingness in the key fields

Two kinds of missingness matter here. The first is *suppression*: Opportunity Insights withholds a tract's mobility estimate when too few children in the birth cohorts grew up there, so the tracts that drop out are systematically the smallest ones. The second is ordinary item missingness in the 2000-era covariates. The table below reports both for every field the analysis uses, on the merged sample before any rows are dropped, and is written to `output/table2_missingness.tex`.


In [7]:
def missingness_table(df, fields):
    """Count and share missing for each field, plus the same for rural tracts only.

    Reported on the merged sample *before* rows are dropped, so the table shows
    what would be lost, not what survived.
    """
    rural_mask = df["popdensity2000"] < utils.RURAL_CUTOFF
    rows = []
    for col, label in fields.items():
        rows.append({
            "field": col,
            "label": label,
            "n_missing": int(df[col].isna().sum()),
            "pct_missing": 100 * df[col].isna().mean(),
            "pct_missing_rural": 100 * df.loc[rural_mask, col].isna().mean(),
        })
    return pd.DataFrame(rows).sort_values("pct_missing", ascending=False)


def write_latex_table(table, out_path, caption_cols):
    """Write a small booktabs table; assumes the caller already ordered the rows."""
    lines = ["\\begin{tabular}{l" + "r" * (len(caption_cols) - 1) + "}", "\\toprule",
             " & ".join(caption_cols) + " \\\\", "\\midrule"]
    for _, r in table.iterrows():
        lines.append("{} & {:,} & {:.1f} & {:.1f} \\\\".format(
            r["label"], r["n_missing"], r["pct_missing"], r["pct_missing_rural"]))
    lines += ["\\bottomrule", "\\end{tabular}"]
    with open(out_path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print("wrote", out_path)

In [8]:
key_fields = {utils.MOBILITY_VAR: "Upward mobility (income rank)",
              "popdensity2000": "Population density, 2000"}
key_fields.update(utils.COVARIATES)

miss = missingness_table(merged_raw, key_fields)
miss[["label", "n_missing", "pct_missing", "pct_missing_rural"]]

,label,n_missing,pct_missing,pct_missing_rural
5,"Annual job growth, 2004-13",2531,3.457698,1.055972
0,Upward mobility (income rank),1189,1.624339,1.635936
7,Mean 3rd-grade math score,1109,1.515048,1.362368
2,Share single-parent households,910,1.243186,0.716748
8,Mean household income,893,1.219962,0.749576
4,Mean commute time,882,1.204934,0.760519
3,Share below poverty line,880,1.202202,0.722219
10,Share with a BA or higher,852,1.163950,0.683920
9,Share of adults employed,851,1.162584,0.683920
11,Share white,827,1.129797,0.552607


In [9]:
os.makedirs(utils.OUT_DIR, exist_ok=True)
write_latex_table(miss, utils.output_path("table2_missingness.tex"),
                  ["Field", "N missing", "\\% missing", "\\% missing (rural)"])

wrote /Users/maxfortner/Documents/Dartmouth/QSS20/qss20-rural-mobility/output/table2_missingness.tex


Missingness runs about 1% on most fields, and — contrary to what the suppression story would predict — it is *lower* among rural tracts than nationally for nearly every covariate (0.6–1.4% vs. 1.1–1.5%). The two exceptions are informative: the Census mail return rate is missing more often in rural tracts (2.5% vs. 0.9%), and population density shows 0.0% rural missingness only mechanically, since a tract has to have a density value to be classified rural in the first place. Mobility itself is suppressed at the same 1.6% rate in both groups. Every model below is fit on complete cases, and the number of tracts dropped is printed with the model.


## Write

In [10]:
out_path = utils.data_path("analysis_sample.csv")
analysis.to_csv(out_path, index=False)
print("wrote {:,} rows x {} cols to {}".format(
    len(analysis), analysis.shape[1], out_path))

wrote 71,958 rows x 42 cols to /Users/maxfortner/Documents/Dartmouth/QSS20/qss20-rural-mobility/data/analysis_sample.csv


In [11]:
analysis[utils.ID_COLS + ["czname", utils.MOBILITY_VAR, "popdensity2000",
                          "density_cat", "above_median_mobility"]].head()

,state,county,tract,czname,kfr_pooled_pooled_p25,popdensity2000,density_cat,above_median_mobility
0,1,1,20100,Montgomery,0.367813,195.72380,Small town\n(100-500),False
1,1,1,20200,Montgomery,0.316781,566.38141,Suburban\n(500-2k),False
2,1,1,20300,Montgomery,0.373485,624.19684,Suburban\n(500-2k),False
3,1,1,20400,Montgomery,0.421511,713.80396,Suburban\n(500-2k),False
4,1,1,20500,Montgomery,0.433415,529.93030,Suburban\n(500-2k),True
